In [1]:
import pandas as pd
import numpy as np

# SciPy contains the core statistical tests (T-tests, Chi-Square)
import scipy.stats as stats

# Statsmodels is excellent for advanced ANOVA and regression modeling
import statsmodels.api as sm
from statsmodels.formula.api import ols


In [2]:
df = pd.read_csv("dataset_cleaned.csv")

print(df[['super_genre', 'popularity_tier', 'popularity']].head())
print(f"\nDataset shape ready for testing: {df.shape}")

  super_genre popularity_tier  popularity
0    Acoustic             Hit          73
1    Acoustic         Average          55
2    Acoustic         Average          57
3    Acoustic             Hit          71
4    Acoustic             Hit          82

Dataset shape ready for testing: (89740, 23)


The Hypotheses for this Test:

Null Hypothesis (H0): The mean popularity is exactly the same across all 15 super genres. (Genre doesn't matter).


Alternative Hypothesis (H1): At least one super genre has a significantly different mean popularity. (Genre absolutely matters).

In [3]:
# Create the Ordinary Least Squares (OLS) model
# We are predicting 'popularity' based on the Categorical variable 'super_genre'
model = ols('popularity ~ C(super_genre)', data=df).fit()

# Generate the ANOVA table
anova_table = sm.stats.anova_lm(model, typ=2)

print("ANOVA TEST RESULTS: POPULARITY BY SUPER GENRE")
print("-" * 50)
display(anova_table)

# Extract and print the p-value specifically
p_value = anova_table.loc['C(super_genre)', 'PR(>F)']
print(f"\nP-Value: {p_value}")

if p_value < 0.05:
    print("Conclusion: Reject the Null Hypothesis. There is a statistically significant difference in popularity between genres.")
else:
    print("Conclusion: Fail to reject the Null Hypothesis. Genre does not significantly impact popularity.")

ANOVA TEST RESULTS: POPULARITY BY SUPER GENRE
--------------------------------------------------


,sum_sq,df,F,PR(>F)
C(super_genre),2.472838e+06,14.0,445.961235,0.0
Residual,3.553726e+07,89725.0,NaN,NaN



P-Value: 0.0
Conclusion: Reject the Null Hypothesis. There is a statistically significant difference in popularity between genres.


In [4]:
import scipy.stats as stats

# Degrees of freedom
df_between = 15 - 1          # 14
df_within = 89740 - 15       # 89,725

# Calculate the critical F-value at the 0.05 significance level
critical_f = stats.f.ppf(1 - 0.05, df_between, df_within)

print(f"The exact Critical F-Value threshold is: {critical_f:.3f}")

The exact Critical F-Value threshold is: 1.692


The F-Statistic represents the ratio of the variance between our genres versus the random noise. Because our F-value was well above the critical threshold, it generated a P-value approaching zero, mathematically proving that the differences we saw in our box plots are statistically significant, proving there is a statistically significant difference in popularity between genres

Since the ANOVA proved that genre matters, the T-Test is how you prove that the actual sonic content (the audio features) matters when separating a Hit from a Flop.

The Hypotheses: 

Null (H0): There is no statistical difference in the average audio feature (e.g., Energy) between Hits and Flops.


Alternative (H1): There is a statistically significant difference.

In [5]:
# Define the specific structural content (audio features) we want to test
audio_features = ['danceability', 'energy', 'valence', 'acousticness', 'loudness', 'tempo']

# Isolate our two extreme groups to compare
hits = df[df['popularity_tier'] == 'Hit']
flops = df[df['popularity_tier'] == 'Flop']

print("INFERENTIAL ANALYSIS: T-TEST RESULTS (HITS vs. FLOPS)")
print("=" * 60)

# Loop through each feature and run a Welch's T-Test (equal_var=False is standard for large datasets)
for feature in audio_features:
    t_stat, p_val = stats.ttest_ind(hits[feature], flops[feature], equal_var=False)
    
    print(f"Feature: {feature.upper()}")
    print(f"  T-Statistic: {t_stat:.3f}")
    
    # Format p-value to avoid ugly scientific notation if it's super small
    if p_val < 0.0001:
        print(f"  P-Value:     < 0.0001")
    else:
        print(f"  P-Value:     {p_val:.4f}")
        
    # Generate business interpretation
    if p_val < 0.05:
        if t_stat > 0:
            print("  Conclusion:  SIGNIFICANT -> Hits have significantly HIGHER levels of this feature than Flops.\n")
        else:
            print("  Conclusion:  SIGNIFICANT -> Hits have significantly LOWER levels of this feature than Flops.\n")
    else:
        print("  Conclusion:  NOT SIGNIFICANT -> No statistical difference between Hits and Flops for this feature.\n")

INFERENTIAL ANALYSIS: T-TEST RESULTS (HITS vs. FLOPS)
Feature: DANCEABILITY
  T-Statistic: 26.948
  P-Value:     < 0.0001
  Conclusion:  SIGNIFICANT -> Hits have significantly HIGHER levels of this feature than Flops.

Feature: ENERGY
  T-Statistic: 6.488
  P-Value:     < 0.0001
  Conclusion:  SIGNIFICANT -> Hits have significantly HIGHER levels of this feature than Flops.

Feature: VALENCE
  T-Statistic: 9.160
  P-Value:     < 0.0001
  Conclusion:  SIGNIFICANT -> Hits have significantly HIGHER levels of this feature than Flops.

Feature: ACOUSTICNESS
  T-Statistic: -20.932
  P-Value:     < 0.0001
  Conclusion:  SIGNIFICANT -> Hits have significantly LOWER levels of this feature than Flops.

Feature: LOUDNESS
  T-Statistic: 31.770
  P-Value:     < 0.0001
  Conclusion:  SIGNIFICANT -> Hits have significantly HIGHER levels of this feature than Flops.

Feature: TEMPO
  T-Statistic: -1.816
  P-Value:     0.0694
  Conclusion:  NOT SIGNIFICANT -> No statistical difference between Hits and Fl